## Creating the Data Set in DuckDB

This notebook outlines the process of creating a dataset of Indian equities using data from the National Stock Exchange (NSE) and market cap information from Yahoo Finance. The dataset is then used to rank stocks by market capitalization.

### Objectives

1. **Fetch NSE Master List**: Retrieve the list of equities from the NSE website.
2. **Clean Data**: Clean and prepare the data for further processing.
3. **Fetch Market Cap**: Use Yahoo Finance to fetch the latest market cap for each equity.
4. **Rank Stocks**: Rank the stocks by market cap and prepare a final dataset.

### Libraries Used

- **pandas**: For data manipulation and analysis.
- **urllib.request**: For fetching data from the NSE website.
- **yfinance**: For fetching market cap data from Yahoo Finance.
- **duckdb**: For efficient data processing and analysis (not used in this section but planned for later steps).
- **tqdm**: For displaying progress bars during data fetching.

### Data Flow

1. **Fetching NSE Master List**: The NSE master list is fetched from the official NSE website and stored in a pandas DataFrame.
2. **Data Cleaning**: The column names are cleaned, and the data is filtered to include only active equities (those with 'EQ' in the SERIES column).
3. **Market Cap Fetching**: For each equity, the latest market cap is fetched from Yahoo Finance using the `yfinance` library.
4. **Ranking Stocks**: The stocks are ranked by market cap, and a final dataset is prepared.

### Code Overview

The code below demonstrates the steps involved in creating the dataset:

1. **Fetching and Cleaning NSE Data**
2. **Fetching Market Cap Data**
3. **Ranking Stocks by Market Cap**

### Notes

- The code uses a safe throttle (`time.sleep(0.1)`) to avoid hitting rate limits when fetching market cap data.
- The final dataset includes the ticker symbol, company name, and market cap in crores (CR).
- The dataset is sorted in descending order of market cap to identify the largest companies by market capitalization.

### Next Steps

- The next step would be to use **DuckDB** for efficient data processing and analysis of the dataset.
- Further enhancements could include adding more financial metrics, visualizations, and exporting the dataset for use in other applications.

### Conclusion

This notebook provides a comprehensive approach to creating a dataset of Indian equities ranked by market capitalization. The dataset can be used for various financial analyses, including portfolio management, investment research, and market trend analysis.

In [2]:
import pandas as pd
import urllib.request
import io

url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
response = urllib.request.urlopen(req)
master_df = pd.read_csv(io.StringIO(response.read().decode('utf-8')))

# Strip any leading/trailing whitespace from NSE column names
master_df.columns = master_df.columns.str.strip()

print("Cleaned Columns:", master_df.columns.tolist())
print(master_df[['SYMBOL', 'SERIES', 'PAID UP VALUE', 'FACE VALUE']].head())

Cleaned Columns: ['SYMBOL', 'NAME OF COMPANY', 'SERIES', 'DATE OF LISTING', 'PAID UP VALUE', 'MARKET LOT', 'ISIN NUMBER', 'FACE VALUE']
       SYMBOL SERIES  PAID UP VALUE  FACE VALUE
0   20MICRONS     EQ              5           5
1  21STCENMGM     BE             10          10
2      360ONE     EQ              1           1
3  3BBLACKBIO     EQ             10          10
4   3IINFOLTD     BE             10          10


In [3]:
import yfinance as yf

t = yf.Ticker("RELIANCE.NS")

print("1. Testing fast_info attribute access:")
try:
    print("   t.fast_info.market_cap =", t.fast_info.market_cap)
except Exception as e:
    print("   FAILED:", type(e), e)

print("\n2. Testing fast_info dictionary access:")
try:
    print("   t.fast_info['market_cap'] =", t.fast_info['market_cap'])
except Exception as e:
    print("   FAILED:", type(e), e)

print("\n3. Testing standard .info dictionary:")
try:
    info = t.info
    print("   t.info.get('marketCap') =", info.get('marketCap'))
    print("   Total keys returned in .info:", len(info))
except Exception as e:
    print("   FAILED:", type(e), e)

1. Testing fast_info attribute access:


c:\Users\chira\AppData\Local\Programs\Python\Python314\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\chira\AppData\Local\Programs\Python\Python314\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\chira\AppData\Local\Programs\Python\Python314\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#depreca

   t.fast_info.market_cap = 18071264287161.633

2. Testing fast_info dictionary access:
   t.fast_info['market_cap'] = 18071264287161.633

3. Testing standard .info dictionary:
   t.info.get('marketCap') = 18071263641600
   Total keys returned in .info: 169


In [4]:
import pandas as pd
import urllib.request
import io

# 1. Load NSE Master List
url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
master_df = pd.read_csv(io.StringIO(urllib.request.urlopen(req).read().decode('utf-8')))

# 2. Clean column names (NSE has leading spaces like ' SERIES')
master_df.columns = master_df.columns.str.strip()

# 3. Filter for active equities ('EQ') and calculate Shares Outstanding
eq_df = master_df[master_df['SERIES'] == 'EQ'].copy()
eq_df['TICKER'] = eq_df['SYMBOL'].astype(str).str.strip() + ".NS"
eq_df['SHARES_OUTSTANDING'] = eq_df['PAID UP VALUE'] / eq_df['FACE VALUE']

print(f"Prepared {len(eq_df)} stocks with exact share counts.")
display(eq_df[['TICKER', 'NAME OF COMPANY', 'PAID UP VALUE', 'FACE VALUE', 'SHARES_OUTSTANDING']].head())

Prepared 2397 stocks with exact share counts.


,TICKER,NAME OF COMPANY,PAID UP VALUE,FACE VALUE,SHARES_OUTSTANDING
0,20MICRONS.NS,20 Microns Limited,5,5,1.0
2,360ONE.NS,360 ONE WAM LIMITED,1,1,1.0
3,3BBLACKBIO.NS,3B Blackbio Dx Limited,10,10,1.0
5,3MINDIA.NS,3M India Limited,10,10,1.0
6,3PLAND.NS,3P Land Holdings Limited,2,2,1.0


In [5]:
import pandas as pd
import yfinance as yf
import duckdb
from datetime import datetime
import time
import warnings
from tqdm import tqdm
import urllib.request
import io

# Suppress Pandas warnings
warnings.filterwarnings("ignore", message=".*inplace keyword.*")
warnings.filterwarnings("ignore")

# 1. Load NSE Master List
url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
master_df = pd.read_csv(io.StringIO(urllib.request.urlopen(req).read().decode('utf-8')))
master_df.columns = master_df.columns.str.strip()

eq_df = master_df[master_df['SERIES'] == 'EQ'].copy()
eq_df['TICKER'] = eq_df['SYMBOL'].astype(str).str.strip() + ".NS"

all_tickers = eq_df['TICKER'].tolist()
market_caps = {}

print("[*] Fetching live market caps directly to ensure accuracy...")
for i, ticker in enumerate(tqdm(all_tickers, desc="Fetching Market Caps")):
    try:
        # Use fast_info which returned the correct market cap in your test
        t = yf.Ticker(ticker)
        mcap = t.fast_info.get('market_cap', 0)
        if not mcap or mcap == 0:
            # Fallback to info dict if fast_info is missing
            mcap = t.info.get('marketCap', 0)
        
        market_caps[ticker] = mcap
        time.sleep(0.1) # Safe throttle
    except Exception:
        market_caps[ticker] = 0.0

# 2. Map back and rank
eq_df['MARKET_CAP'] = eq_df['TICKER'].map(market_caps).fillna(0.0)
eq_df['MARKET_CAP_CR'] = (eq_df['MARKET_CAP'] / 1e7).round(2)

ranked_universe = eq_df[eq_df['MARKET_CAP'] > 0].sort_values(by='MARKET_CAP', ascending=False).reset_index(drop=True)

print(f"\n[✓] Ranked {len(ranked_universe)} active NSE stocks successfully!")
print(f"    #1 Ranked Stock:   {ranked_universe.loc[0, 'TICKER']} ({ranked_universe.loc[0, 'NAME OF COMPANY']}) - ₹{ranked_universe.loc[0, 'MARKET_CAP_CR']:,.2f} Cr")
print(f"    #500 Cutoff Cap:   {ranked_universe.loc[499, 'TICKER']} ({ranked_universe.loc[499, 'NAME OF COMPANY']}) - ₹{ranked_universe.loc[499, 'MARKET_CAP_CR']:,.2f} Cr")

display(ranked_universe[['TICKER', 'NAME OF COMPANY', 'MARKET_CAP_CR']].head(10))

[*] Fetching live market caps directly to ensure accuracy...


Fetching Market Caps: 100%|██████████| 2397/2397 [23:44<00:00,  1.68it/s]


[✓] Ranked 2345 active NSE stocks successfully!
    #1 Ranked Stock:   RELIANCE.NS (Reliance Industries Limited) - ₹1,807,126.36 Cr
    #500 Cutoff Cap:   PGIL.NS (Pearl Global Industries Limited) - ₹12,857.60 Cr


,TICKER,NAME OF COMPANY,MARKET_CAP_CR
0,RELIANCE.NS,Reliance Industries Limited,1807126.36
1,BHARTIARTL.NS,Bharti Airtel Limited,1316494.40
2,JPL.NS,Jio Platforms Limited,1292487.47
3,HDFCBANK.NS,HDFC Bank Limited,1244768.76
4,ICICIBANK.NS,ICICI Bank Limited,1128542.61
5,SBIN.NS,State Bank of India,979553.07
6,TCS.NS,Tata Consultancy Services Limited,894427.46
7,BAJFINANCE.NS,Bajaj Finance Limited,671434.77
8,LT.NS,Larsen & Toubro Limited,579523.68
9,LICI.NS,Life Insurance Corporation Of India,505999.79


In [ ]:
# Change the db path to something more sensible like NSE-universe-raw
db_path = "NSE-universe-raw.duckdb"
conn = duckdb.connect(db_path)
# Add metadata columns to your existing in-memory DataFrame
ranked_universe[' MARKET_CAP_RANK'] = ranked_universe.index + 1
ranked_universe['SYNC_WEEK'] = datetime.now().strftime("%Y-W%V")
ranked_universe['UPDATED_AT'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
# Create persistent Universe Ranking Table and insert directly from Pandas
conn.execute("""
CREATE TABLE IF NOT EXISTS nse_universe_ranks (
ticker VARCHAR PRIMARY KEY,
symbol VARCHAR,
company_name VARCHAR,
isin VARCHAR,
face_value DOUBLE,
market_cap DOUBLE,
market_cap_cr DOUBLE,
market_cap_rank INTEGER,
sync_week VARCHAR,
updated_at VARCHAR
)
""")
conn.execute("""
INSERT OR REPLACE INTO nse_universe_ranks
SELECT 
TICKER,
SYMBOL,
"NAME OF COMPANY",
"ISIN NUMBER",
"FACE VALUE",
MARKET_CAP,
MARKET_CAP_CR,
" MARKET_CAP_RANK",
SYNC_WEEK,
UPDATED_AT
FROM ranked_universe
""")
count = conn.execute("SELECT COUNT(*) FROM nse_universe_ranks").fetchone()[0]
print(f"[✓] Saved {count} ranked NSE equities into 'nse_universe_ranks' in {db_path}!")
conn.close()

[✓] Saved 2345 ranked NSE equities into 'nse_universe_ranks' in NSE-universe-raw.duckdb!
